In [1]:
from rapidsegment import *

In [2]:



my_search_grid = {
    "min_sample_size": [10000,5000, 1000],  # Example values for min_sample_size   
    "min_lift": [2.0,1.5]
}


# Initialize the builder
builder = StrategicSegmentBuilder(
    target="is_defaulted",
    # n_jobs=-1,
    min_sample_size=1000,
    min_lift=1.1,
    min_events = 5,
    top_n_vars=15,
    max_segments=10,
    enable_diversity=True,
    max_feature_reuse=1,
    enable_1way=True,  # Include 1-way rules from final output
    enable_2way=True,  # Exclude 2-way rules from final output
    enable_3way=True,  # Exclude 3-way rules from final output
    feature_groups=None,
    ignore_features=['application_id'],
    param_grid=my_search_grid
)


data = UniversalDataLoader(file_path=r"/workspaces/RapidSegment/loan_applications_500k.parquet").load()

segments_df = builder.extract_segments(data)
final_eval = builder.evaluate_final_coverage(data)

2026-07-17 16:04:39,603 | INFO     | [builder.py:329] | DuckDB Configured: Threads=4/4, MemoryLimit=7GB
2026-07-17 16:04:40,146 | INFO     | [builder.py:363] | Iteration 1 | Remaining Volume: 500,000 | Base Rate: 15.01%
2026-07-17 16:04:47,019 | INFO     | [builder.py:514] | Feature Usage Tracker Update -> 'is_auto_pay' used count = 1
2026-07-17 16:04:47,020 | INFO     | [builder.py:530] | Segment 1 Captured (Size Floor: 10000 | Lift Floor: 2.0): rows=187971, events=60151.0, lift=2.13
  Rule: is_auto_pay=(-inf, 0.50)
  SQL: is_auto_pay < 0.50
2026-07-17 16:04:47,659 | INFO     | [builder.py:363] | Iteration 2 | Remaining Volume: 312,029 | Base Rate: 4.77%
2026-07-17 16:04:51,206 | INFO     | [builder.py:514] | Feature Usage Tracker Update -> 'annual_income' used count = 1
2026-07-17 16:04:51,207 | INFO     | [builder.py:530] | Segment 2 Captured (Size Floor: 10000 | Lift Floor: 2.0): rows=50678, events=5118.0, lift=2.12
  Rule: annual_income=(-inf, 10000.13)
  SQL: annual_income < 1000

In [3]:
from prettytable import PrettyTable
import pandas as pd
table = PrettyTable()
table.field_names = list(pd.DataFrame(final_eval).columns)
for _, row in pd.DataFrame(final_eval).iterrows():
    table.add_row(list(row))
print(table)

+---------+-------------+---------------+--------------------+--------------------+--------------+---------------------+
| segment | total_count | target_events |   response_rate    | base_response_rate | capture_rate |         lift        |
+---------+-------------+---------------+--------------------+--------------------+--------------+---------------------+
|   0.0   |   225045.0  |     5103.0    | 2.2675464907018594 | 15.009400000000001 |    45.009    | 0.15107509232226865 |
|   1.0   |   187971.0  |    60151.0    | 32.00014895914796  | 15.009400000000001 |   37.5942    |  2.1320072060940447 |
|   2.0   |   50678.0   |     5118.0    | 10.099056789928568 | 15.009400000000001 |   10.1356    |  0.6728488007467698 |
|   3.0   |   23568.0   |     4044.0    | 17.158859470468432 | 15.009400000000001 |    4.7136    |  1.143207554630327  |
|   4.0   |   12738.0   |     631.0     | 4.953681896687078  | 15.009400000000001 |    2.5476    | 0.33003863556751617 |
+---------+-------------+-------

In [4]:
print("--- FULL SEGMENT RULES ---\n")

for index, row in pd.DataFrame(segments_df).iterrows():
    print(f"Segment ID: {row['segment_id']}")
    print(f"Raw Rule:   {row['rule_string']}")
    print(f"SQL Filter: {row['sql_filter']}")
    print("-" * 50)

--- FULL SEGMENT RULES ---

Segment ID: 1
Raw Rule:   is_auto_pay=(-inf, 0.50)
SQL Filter: is_auto_pay < 0.50
--------------------------------------------------
Segment ID: 2
Raw Rule:   annual_income=(-inf, 10000.13)
SQL Filter: annual_income < 10000.13
--------------------------------------------------
Segment ID: 3
Raw Rule:   credit_score=(-inf, 609.01)
SQL Filter: credit_score < 609.01
--------------------------------------------------
Segment ID: 4
Raw Rule:   debt_to_income_ratio=[50.87, 58.35)
SQL Filter: (debt_to_income_ratio >= 50.87 AND debt_to_income_ratio < 58.35)
--------------------------------------------------


In [5]:
builder.explain_feature_journey("recovery_collection_fee")

AUDIT TRAIL FOR FEATURE: 'recovery_collection_fee'

[Iteration 1]
  • Current dynamic IV   : 0.0000
  • Previous times used  : 0
  • Selection Status     : Excluded (Information Value is Zero/Invalid)
  • Winner this round    : is_auto_pay=(-inf, 0.50) (Variables: ['is_auto_pay'])

[Iteration 2]
  • Current dynamic IV   : 0.0000
  • Previous times used  : 0
  • Selection Status     : Excluded (Information Value is Zero/Invalid)
  • Winner this round    : annual_income=(-inf, 10000.13) (Variables: ['annual_income'])

[Iteration 3]
  • Current dynamic IV   : 0.0000
  • Previous times used  : 0
  • Selection Status     : Excluded (Information Value is Zero/Invalid)
  • Winner this round    : credit_score=(-inf, 609.01) (Variables: ['credit_score'])

[Iteration 4]
  • Current dynamic IV   : 0.0000
  • Previous times used  : 0
  • Selection Status     : Excluded (Information Value is Zero/Invalid)
  • Winner this round    : debt_to_income_ratio=[50.87, 58.35) (Variables: ['debt_to_income_ra